# Sentiment Analysis Model
Training Naive Bayes and Logistic Regression models for sentiment classification

In [ ]:
# Install required libraries
!pip install scikit-learn pandas numpy nltk joblib -q

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import re

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

In [ ]:
# Create sample training data
data = {
    'review': [
        'This product is excellent and amazing!',
        'Love it, best purchase ever',
        'Very satisfied with quality',
        'Absolutely wonderful',
        'It is okay, nothing special',
        'Neither good nor bad',
        'Average product',
        'Terrible quality, waste of money',
        'Very disappointed',
        'Worst purchase ever',
    ],
    'sentiment': [2, 2, 2, 2, 1, 1, 1, 0, 0, 0]  # 0=Negative, 1=Neutral, 2=Positive
}

df = pd.DataFrame(data)
print('Dataset shape:', df.shape)
print('\nSentiment distribution:')
sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for idx, count in df['sentiment'].value_counts().sort_index().items():
    print(f'{sentiment_map[idx]}: {count}')

In [ ]:
# Text Preprocessing
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(token) for token in tokens if token not in stop_words]
    return ' '.join(tokens)

df['review_processed'] = df['review'].apply(preprocess_text)
print('Text preprocessing complete')

In [ ]:
# Feature Extraction
vectorizer = TfidfVectorizer(max_features=5000, min_df=1, max_df=0.95)
X = vectorizer.fit_transform(df['review_processed'])
y = df['sentiment']

print('Feature matrix shape:', X.shape)
print('Number of features:', X.shape[1])

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set: {X_test.shape[0]} samples')

In [ ]:
# Train Naive Bayes (Recommended)
print('Training Naive Bayes Classifier...')
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print('\n=== Naive Bayes Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}')
print(f'Precision (macro): {precision_score(y_test, y_pred_nb, average="macro"):.4f}')
print(f'Recall (macro): {recall_score(y_test, y_pred_nb, average="macro"):.4f}')
print(f'F1-Score (macro): {f1_score(y_test, y_pred_nb, average="macro"):.4f}')

In [ ]:
# Train Logistic Regression
print('Training Logistic Regression Classifier...')
lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print('\n=== Logistic Regression Performance ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision (macro): {precision_score(y_test, y_pred_lr, average="macro"):.4f}')
print(f'Recall (macro): {recall_score(y_test, y_pred_lr, average="macro"):.4f}')
print(f'F1-Score (macro): {f1_score(y_test, y_pred_lr, average="macro"):.4f}')

In [ ]:
# Save models
import os
os.makedirs('/content/models', exist_ok=True)

joblib.dump(nb_model, '/content/models/sentiment_model.pkl')
joblib.dump(vectorizer, '/content/models/vectorizer_sentiment.pkl')

print('✓ Naive Bayes model saved')
print('✓ Vectorizer saved')

In [ ]:
# Test predictions
test_reviews = [
    'This is absolutely wonderful!',
    'Decent product, nothing special',
    'Terrible experience, very disappointed',
]

print('=== Test Predictions ===\n')

for review in test_reviews:
    processed = preprocess_text(review)
    X_review = vectorizer.transform([processed])
    
    prediction = nb_model.predict(X_review)[0]
    probability = nb_model.predict_proba(X_review)[0]
    
    sentiment_label = sentiment_map[prediction]
    confidence = max(probability)
    
    print(f'Review: {review}')
    print(f'Sentiment: {sentiment_label} (Confidence: {confidence:.2%})\n')